Training a model on my dataset

In [1]:
%pip install transformers[torch]

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import AutoTokenizer , AutoModelForSequenceClassification , DataCollatorWithPadding
import torch
from datasets import load_dataset


C:\ProgramData\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("glue","mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [4]:
def tokenize_function(examples):
    return tokenizer(examples['sentence1'], examples['sentence2'], truncation = True )

In [5]:
tokenized_datasets = dataset.map(tokenize_function, batched = True)
data_collator = DataCollatorWithPadding(tokenizer)

In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")

model = AutoModelForSequenceClassification.from_pretrained(checkpoint , num_labels = True)

C:\Users\Lenovo\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2810.15it/s]
[transformers] BertForSequenceClassif

In [9]:
from transformers import Trainer 

trainer = Trainer(
    model,
    training_args,
    train_dataset = tokenized_datasets['train'],
    eval_dataset = tokenized_datasets['validation'],
    data_collator = data_collator, 
    processing_class = tokenizer,
)

In [10]:
trainer.train()

C:\Users\Lenovo\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.195648
1000,0.095304


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.81s/it]
C:\Users\Lenovo\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.11s/it]


TrainOutput(global_step=1377, training_loss=0.11706243430520627, metrics={'train_runtime': 7968.802, 'train_samples_per_second': 1.381, 'train_steps_per_second': 0.173, 'total_flos': 405111332351112.0, 'train_loss': 0.11706243430520627, 'epoch': 3.0})

In [12]:
%pip install evaluate 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [14]:

predictions = trainer.predict(tokenized_datasets['validation'])


import numpy as np 

preds = np.argmax(predictions.predictions, axis = -1)


    

In [17]:
%pip install scikit-learn scipy

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.0 MB 3.0 MB/s eta 0:00:03
   ------- -------------------------------- 1.6/8.0 MB 4.5 MB/s eta 0:00:02
   ------------- -------------------------- 2.6/8.0 MB 4.6 MB/s eta 0:00:02
   ----------------- ---------------------- 3.4/8.0 MB 4.4 MB/s eta 0:00:02
   ---------------------- ----------------- 4.5/8.0 MB 4.5 MB/s eta 0:00:01
   --------------------------- ------------ 5.5/8.0 MB 4.6 MB/s eta 0:00:01
   -------------------------------- ------- 6.6/8.0 MB 4.7 MB/s eta 0:00:01
   ------------------------------------ --- 7.3/8.0 MB 4.7 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 4.4 MB/s eta 0:00:00

   ---------------------------------------- 0/3 [threadpoolctl]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3

In [18]:
import evaluate 

metric = evaluate.load("glue","mrpc")

metric.compute(predictions = preds , references = predictions.label_ids)

def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits , labels = eval_preds
    predictions = np.argmax(logits , axis = -1)
    return metric.compute(predictions = predictions , references = predictions.label_ids)
    

EvaluationModule(name: "glue", module_type: "metric", features: {'predictions': Value('int64'), 'references': Value('int64')}, usage: """
Compute GLUE evaluation metric associated to each GLUE dataset.
Args:
    predictions: list of predictions to score.
        Each translation should be tokenized into a list of tokens.
    references: list of lists of references for each translation.
        Each reference should be tokenized into a list of tokens.
Returns: depending on the GLUE subset, one or several of:
    "accuracy": Accuracy
    "f1": F1 score
    "pearson": Pearson Correlation
    "spearmanr": Spearman Correlation
    "matthews_correlation": Matthew Correlation
Examples:

    >>> glue_metric = evaluate.load('glue', 'sst2')  # 'sst2' or any of ["mnli", "mnli_mismatched", "mnli_matched", "qnli", "rte", "wnli", "hans"]
    >>> references = [0, 1]
    >>> predictions = [0, 1]
    >>> results = glue_metric.compute(predictions=predictions, references=references)
    >>> print(results